In [17]:
%pip install -r req.txt

Processing /work/perseverance-python-buildout/croot/joblib_1728392999985/work (from -r req.txt (line 40))
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/work/perseverance-python-buildout/croot/joblib_1728392999985/work'

Note: you may need to restart the kernel to use updated packages.


<h2> tokenizer <h2\>

In [18]:
import torch

class CharTokenizer:
    def __init__(self, text=None, pad_token='<PAD>', unk_token='<UNK>'):
        self.pad_token = pad_token
        self.unk_token = unk_token
        self.char2idx = {}
        self.idx2char = {}
        self.vocab_built = False

        if text:
            self.build_vocab(text)
        
        self.vocab_size = len(self.char2idx)

    def build_vocab(self, text):
        unique_chars = sorted(set(text))
        # Reserve indices for PAD and UNK
        self.char2idx = {self.pad_token: 0, self.unk_token: 1}
        for i, ch in enumerate(unique_chars, start=2):
            self.char2idx[ch] = i
        self.idx2char = {i: ch for ch, i in self.char2idx.items()}
        self.vocab_built = True

    def encode(self, text, max_length=None):
        if not self.vocab_built:
            raise ValueError("Vocabulary not built yet. Call build_vocab first.")

        encoded = [self.char2idx.get(ch, self.char2idx[self.unk_token]) for ch in text]

        # Truncate if needed
        if max_length is not None:
            encoded = encoded[:max_length]

        # Pad if needed
        if max_length is not None and len(encoded) < max_length:
            pad_length = max_length - len(encoded)
            encoded += [self.char2idx[self.pad_token]] * pad_length

        return torch.tensor(encoded, dtype=torch.long)

    def decode(self, indices):
        # Accept either tensor or list
        if isinstance(indices, torch.Tensor):
            indices = indices.tolist()
        chars = [self.idx2char.get(i, self.unk_token) for i in indices]
        # Strip padding tokens from end
        while chars and chars[-1] == self.pad_token:
            chars.pop()
        return ''.join(chars)
    
    def create_mask(self, encoded_tensor):
        pad_token_idx = self.char2idx[self.pad_token]
        return (encoded_tensor != pad_token_idx).long()


<h2> DataSet <h2\>

In [19]:
from torch.utils.data import Dataset
import pandas as pd

class CISC2010DataSet(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=1024):
        self.df = dataframe.drop_duplicates(subset='content', keep='first')
        self.df = self.df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        text = self.df.loc[idx]['content']
        encoded = self.tokenizer.encode(text, max_length=self.max_length)
        label = torch.tensor(self.df.loc[idx]['classification'], dtype=torch.long)
        length = min(len(text), self.max_length)
        
        return encoded, length, label

<h2> The Model <h2\>

In [20]:
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class LSTMBased(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim, device, num_layers = 1, dropout = 0):
        super().__init__()
        self.device = device
        self.embedding = nn.Embedding(input_dim, hidden_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout ,
            bidirectional=True
        )
        self.ffnn = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim * 8),
            nn.ReLU(),
            nn.Linear(hidden_dim * 8, output_dim)
            )
    
    def forward(self, x, lengths):
        embedded = self.embedding(x)
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=True).to(self.device)
        packed_out, (h_n, c_n) = self.lstm(packed)
        out, _ = pad_packed_sequence(packed_out, batch_first=True)

        # Gather last valid time-step for each sequence
        idx = (lengths - 1).unsqueeze(1).unsqueeze(2).expand(-1, 1, out.size(2))
        last_outputs = out.gather(1, idx).squeeze(1)

        return self.ffnn(last_outputs)


<h2> datalodaer <h2\>

In [21]:

def collect_fn(batch):

    encodedSeqs, lengths, labels = zip(*batch)

    encodedSeqs = torch.stack(encodedSeqs)
    lengths = torch.tensor(lengths, dtype=torch.long)
    labels = torch.tensor(labels, dtype=torch.long)

    lengths, perm_idx = lengths.sort(descending=True)
    encodedSeqs = encodedSeqs[perm_idx]
    labels = labels[perm_idx]
    
    return encodedSeqs, lengths, labels

<h2>train function<h2\>

In [22]:
from tqdm.auto import tqdm
import os

def train(model,
              dataloader,
              optimizer,
              device,
              loss_function,
              epochs: int = 1000,
              save_every: int = 100,
              save_dir: str = "checkpoints"):
    os.makedirs(save_dir, exist_ok=True)
    model.to(device)

    start_epoch = 1
    ckpts = [f for f in os.listdir(save_dir) if f.endswith(".pt")]
    if ckpts:
        latest_ckpt = max(ckpts, key=lambda x: int(x.split("_")[-1].split(".")[0]))
        ckpt_path = os.path.join(save_dir, latest_ckpt)
        checkpoint = torch.load(ckpt_path, map_location=device)
        
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        start_epoch = checkpoint["epoch"] + 1
        
        tqdm.write(f"✅ Loaded checkpoint '{ckpt_path}' (epoch {checkpoint['epoch']})")
    

    epoch_bar = tqdm(range(start_epoch, epochs + 1), desc="Epochs", unit="epoch")

    for epoch in epoch_bar:
        model.train()
        running_loss = 0.0

        for input, lengths, labels in dataloader:

            input = input.to(device)
            labels = labels.to(device)
            lengths = lengths.to(device)

            optimizer.zero_grad()
            logits = model(input, lengths)
            loss = loss_function(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_loss = running_loss / len(dataloader)
        epoch_bar.set_postfix(avg_loss=f"{avg_loss:.4f}")

        # ── checkpoint every `save_every` epochs ───────────────────────────
        if epoch % save_every == 0:
            ckpt_path = os.path.join(save_dir, f"pln_epoch_{epoch}.pt")
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                },
                ckpt_path,
            )
            tqdm.write(f"✓ Saved checkpoint → {ckpt_path}")
        


<h2> eval function <h2\>

In [23]:
from sklearn.metrics import precision_score, recall_score, f1_score
from tqdm import tqdm
import torch.nn.functional as F
import random

def evaluate(model, dataloader, device, tokenizer, max_samples):
    model.eval()
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    correct_samples = []
    incorrect_samples = []

    batch_bar = tqdm(enumerate(dataloader), total=len(dataloader), unit="batch", leave=False)
    with torch.no_grad():
        for batch_idx, (inputs, lengths, labels) in batch_bar:
            inputs = inputs.to(device)
            labels = labels.to(device)
            lengths = lengths.to(device)
        
            batch_bar.set_description(f"Batch {batch_idx}")

            logits = model(inputs, lengths)
            probs = F.softmax(logits, dim=-1)
            preds = probs.argmax(dim=-1)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

            correct_mask = (preds == labels)
            correct += correct_mask.sum().item()
            total += labels.size(0)

            for i in range(len(labels)):
                if len(correct_samples) >= max_samples and len(incorrect_samples) >= max_samples:
                    break
                decoded_input = tokenizer.decode(inputs[i].cpu())
                true_label = labels[i].item()
                pred_label = preds[i].item()

                sample = {
                    'input': decoded_input,
                    'true_label': true_label,
                    'pred_label': pred_label,
                }

                if pred_label == true_label and len(correct_samples) < max_samples:
                    if random.random() < 0.05:
                        correct_samples.append(sample)
                elif pred_label != true_label and len(incorrect_samples) < max_samples:
                    if random.random() < 0.05:
                        incorrect_samples.append(sample)

            if len(correct_samples) >= max_samples and len(incorrect_samples) >= max_samples:
                break

    accuracy = correct / total if total > 0 else 0
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)


    return accuracy, precision, recall, f1, correct_samples, incorrect_samples

<h1> All comes together <h1\>

In [24]:
from torch.utils.data import DataLoader
import torch.optim as optim

def main():
    max_length = 100
    # Load dataset
    train_csv_path = "CISC2010_cleaned_train.csv"
    train_df = pd.read_csv(train_csv_path)

    test_csv_path = "CISC2010_cleaned_train.csv"
    test_df = pd.read_csv(test_csv_path)

    # Build tokenizer vocab from all content
    tokenizer = CharTokenizer("".join(train_df["content"].tolist()))

    # Dataset + DataLoader
    train_dataset = CISC2010DataSet(train_df, tokenizer, max_length=max_length)
    train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collect_fn)

    test_dataset = CISC2010DataSet(test_df, tokenizer, max_length=max_length)
    test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collect_fn)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"running on {device}")

    # Init model
    vocab_size = tokenizer.vocab_size
    embedding_dim = 64
    model = LSTMBased(input_dim=vocab_size, hidden_dim=embedding_dim, output_dim=2, num_layers = 2, dropout = 0.0, device=device)

    # Optimizer
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    # Train
    train(model=model,
              dataloader=train_dataloader,
              optimizer=optimizer,
              device=device,
              loss_function=F.cross_entropy,
              epochs = 400,
              save_every = 50,
              save_dir = "checkpointsDROP0Layer2")
    
    accuracy, precision, recall, f1, correct_samples, incorrect_samples = evaluate(
        model=model,
        dataloader=test_dataloader,
        device=device,
        tokenizer=tokenizer,
        max_samples=10
        )
    
    print("\n====== Final Evaluation Metrics ======")
    print(f"Accuracy   : {accuracy:.4f}")
    print(f"Precision  : {precision:.4f}")
    print(f"Recall     : {recall:.4f}")
    print(f"F1 Score   : {f1:.4f}")
    print("======================================\n")

    print("Sample Correct Predictions:")
    for sample in correct_samples:
        print(f"Input      : {sample['input']}")
        print(f"True Label : {sample['true_label']}, Predicted: {sample['pred_label']}")
        print("-----")

    print("\nSample Incorrect Predictions:")
    for sample in incorrect_samples:
        print(f"Input      : {sample['input']}")
        print(f"True Label : {sample['true_label']}, Predicted: {sample['pred_label']}")
        print("-----")

if __name__ == "__main__":
    main()

running on cuda
✅ Loaded checkpoint 'checkpointsDROP0Layer2/pln_epoch_400.pt' (epoch 400)


Epochs: 0epoch [00:00, ?epoch/s]



====== Final Evaluation Metrics ======
Accuracy   : 0.9820
Precision  : 0.9983
Recall     : 0.9739
F1 Score   : 0.9860

Sample Correct Predictions:
Input      : http://localhost:8080/tienda1/miembros/editar.jsp?modo=registro&login=bolduc&password=p%FAblica&nomb
True Label : 0, Predicted: 0
-----
Input      : http://localhost:8080/tienda1/publico/caracteristicas.jsp?id=1 HTTP/1.1 
True Label : 0, Predicted: 0
-----
Input      : http://localhost:8080/tienda1/miembros/index.jsp HTTP/1.1 
True Label : 0, Predicted: 0
-----
Input      : http://localhost:8080/tienda1/publico/autenticar.jsp?modo=entrar&login=diffee&pwd=Gatada&remember=on
True Label : 0, Predicted: 0
-----
Input      : http://localhost:8080/tienda1/miembros/imagenes/castro.jpg/4861362529278789730 HTTP/1.1 
True Label : 1, Predicted: 1
-----
Input      : http://localhost:8080/tienda1/imagenes/2.gif HTTP/1.1 
True Label : 0, Predicted: 0
-----
Input      : http://localhost:8080/tienda1/miembros/editar.jsp HTTP/1.1 modo=registro